# NetShield — Phase 8: Domain Adaptation for Cross-Dataset Intrusion Detection

## Goal
Can we improve cross-dataset intrusion detection by reducing source–target distribution mismatch while keeping target labels completely out of training and tuning?

This phase builds on Phases 1–7:
- strong within-dataset performance;
- severe cross-dataset degradation;
- large feature/traffic distribution differences;
- semantic alignment alone was insufficient.

### Primary intervention
**CORAL (Correlation Alignment)** — an unsupervised domain-alignment method.

### Controlled comparison
1. Common semantic baseline
2. CORAL
3. CORAL + class weighting

Transfer directions:
- UNSW → CICIDS
- CICIDS → UNSW

Target labels are used only for final evaluation.


In [ ]:
from pathlib import Path
import json, warnings, gc
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
UNSW_DIR = RAW_DIR / "UNSW-NB15"
CICIDS_DIR = RAW_DIR / "CICIDS2017"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "domain_adaptation"
TABLE_DIR = OUTPUT_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


In [ ]:
def find_file(root, filename):
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root}")
    return matches[0]

unsw_train = pd.read_csv(find_file(UNSW_DIR, "UNSW_NB15_training-set.csv"))
unsw_test = pd.read_csv(find_file(UNSW_DIR, "UNSW_NB15_testing-set.csv"))
unsw_train.columns = unsw_train.columns.astype(str).str.strip()
unsw_test.columns = unsw_test.columns.astype(str).str.strip()

unsw_train["label_binary"] = pd.to_numeric(unsw_train["label"], errors="coerce").astype("int8")
unsw_test["label_binary"] = pd.to_numeric(unsw_test["label"], errors="coerce").astype("int8")

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)


In [ ]:
CICIDS_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infiltration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

def find_cicids_file(filename):
    matches = list(CICIDS_DIR.rglob(filename))
    if matches:
        return matches[0]
    if "Infiltration" in filename:
        alt = filename.replace("Infiltration", "Infilteration")
        matches = list(CICIDS_DIR.rglob(alt))
        if matches:
            return matches[0]
    raise FileNotFoundError(filename)

parts = []
for fname in CICIDS_FILES:
    part = pd.read_csv(find_cicids_file(fname), low_memory=False)
    part.columns = part.columns.astype(str).str.strip()
    parts.append(part)
    print(fname, part.shape)

cicids = pd.concat(parts, ignore_index=True)
cicids["label_binary"] = (
    cicids["Label"].astype("string").str.strip().str.upper().ne("BENIGN").astype("int8")
)
del parts
gc.collect()

print("CICIDS:", cicids.shape)


In [ ]:
COMMON_FEATURES = [
    "flow_duration_sec","fwd_packets","bwd_packets","fwd_bytes","bwd_bytes",
    "total_packets","total_bytes","packet_rate","fwd_packet_rate",
    "bwd_packet_rate","avg_packet_size"
]

def safe_divide(num, den):
    num = pd.to_numeric(num, errors="coerce").astype("float32")
    den = pd.to_numeric(den, errors="coerce").astype("float32").mask(lambda x: x <= 0)
    return num / den

def build_unsw_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce").astype("float32")
    out["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce").astype("float32")
    out["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce").astype("float32")
    out["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce").astype("float32")
    out["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce").astype("float32")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = safe_divide(out["total_packets"], out["flow_duration_sec"])
    out["fwd_packet_rate"] = safe_divide(out["fwd_packets"], out["flow_duration_sec"])
    out["bwd_packet_rate"] = safe_divide(out["bwd_packets"], out["flow_duration_sec"])
    out["avg_packet_size"] = safe_divide(out["total_bytes"], out["total_packets"])
    return out.replace([np.inf,-np.inf], np.nan).astype("float32")

def build_cicids_common(df):
    out = pd.DataFrame(index=df.index)
    out["flow_duration_sec"] = pd.to_numeric(df["Flow Duration"], errors="coerce").astype("float32") / np.float32(1_000_000)
    out["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce").astype("float32")
    out["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce").astype("float32")
    out["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce").astype("float32")
    out["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce").astype("float32")
    out["total_packets"] = out["fwd_packets"] + out["bwd_packets"]
    out["total_bytes"] = out["fwd_bytes"] + out["bwd_bytes"]
    out["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce").astype("float32")
    out["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce").astype("float32")
    out["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce").astype("float32")
    out["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce").astype("float32")
    return out.replace([np.inf,-np.inf], np.nan).astype("float32")

unsw_common_train = build_unsw_common(unsw_train)
unsw_common_test = build_unsw_common(unsw_test)
cicids_common = build_cicids_common(cicids)

print(unsw_common_train.shape, unsw_common_test.shape, cicids_common.shape)


In [ ]:
# Unlabeled CICIDS sample used only to estimate target statistics for CORAL.
# Evaluation remains on the complete CICIDS set.
CORAL_TARGET_SAMPLE = 300_000
rng = np.random.default_rng(RANDOM_STATE)
cicids_alignment_idx = rng.choice(
    len(cicids_common),
    size=min(CORAL_TARGET_SAMPLE, len(cicids_common)),
    replace=False
)
cicids_alignment = cicids_common.iloc[cicids_alignment_idx]

def medians(X):
    return X[COMMON_FEATURES].median()

def fill(X, m):
    return X[COMMON_FEATURES].fillna(m).astype("float32")

unsw_m = medians(unsw_common_train)
cicids_m = medians(cicids_alignment)

X_unsw_train = fill(unsw_common_train, unsw_m)
X_unsw_test = fill(unsw_common_test, unsw_m)
X_cicids_all = fill(cicids_common, cicids_m)
X_cicids_align = fill(cicids_alignment, cicids_m)

y_unsw_train = unsw_train["label_binary"].to_numpy()
y_unsw_test = unsw_test["label_binary"].to_numpy()
y_cicids = cicids["label_binary"].to_numpy()
y_cicids_train = y_cicids[cicids_alignment_idx]

print("CICIDS alignment sample:", len(cicids_alignment_idx))


In [ ]:
def matrix_power(A, power, eps=1e-5):
    vals, vecs = np.linalg.eigh(A)
    vals = np.clip(vals, eps, None)
    return (vecs * (vals ** power)) @ vecs.T

def coral_fit_transform(X_source, X_target):
    Xs = np.asarray(X_source, dtype=np.float64)
    Xt = np.asarray(X_target, dtype=np.float64)
    ms, mt = Xs.mean(axis=0), Xt.mean(axis=0)
    Cs = np.cov(Xs-ms, rowvar=False) + 1e-4*np.eye(Xs.shape[1])
    Ct = np.cov(Xt-mt, rowvar=False) + 1e-4*np.eye(Xt.shape[1])
    A = matrix_power(Cs, -0.5) @ matrix_power(Ct, 0.5)
    return (Xs-ms) @ A + mt, {"A": A, "source_mean": ms, "target_mean": mt}

def apply_source_alignment(X, stats):
    return (np.asarray(X, dtype=np.float64)-stats["source_mean"]) @ stats["A"] + stats["target_mean"]

# NOTE:
# This implementation estimates the alignment using unlabeled target features.
# The target evaluation labels are never used in fitting.


In [ ]:
MODELS = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}

WEIGHTED_MODELS = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}

def evaluate(model, Xtr, ytr, Xte, yte, direction, method, name):
    m = clone(model)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    score = m.predict_proba(Xte)[:,1]
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0,1]).ravel()
    return {
        "direction":direction, "method":method, "model":name,
        "precision":precision_score(yte,pred,zero_division=0),
        "recall":recall_score(yte,pred,zero_division=0),
        "f1":f1_score(yte,pred,zero_division=0),
        "macro_f1":f1_score(yte,pred,average="macro",zero_division=0),
        "pr_auc":average_precision_score(yte,score),
        "roc_auc":roc_auc_score(yte,score),
        "fpr":fp/(fp+tn) if fp+tn else np.nan,
        "fnr":fn/(fn+tp) if fn+tp else np.nan,
        "tn":tn,"fp":fp,"fn":fn,"tp":tp
    }, m


In [ ]:
# ============================================================
# Baseline: common semantic representation
# ============================================================
rows = []

for name, model in MODELS.items():
    r, _ = evaluate(
        model, X_unsw_train, y_unsw_train, X_cicids_all, y_cicids,
        "UNSW_to_CICIDS", "common_baseline", name
    )
    rows.append(r)

for name, model in MODELS.items():
    r, _ = evaluate(
        model, X_cicids_align, y_cicids_train, X_unsw_test, y_unsw_test,
        "CICIDS_to_UNSW", "common_baseline", name
    )
    rows.append(r)

baseline = pd.DataFrame(rows)
display(baseline)


In [ ]:
# ============================================================
# CORAL: UNSW → CICIDS
# ============================================================
Xs_aligned, coral_stats_u2c = coral_fit_transform(
    X_unsw_train.to_numpy(), X_cicids_align.to_numpy()
)

# Target remains in target coordinates; the classifier is trained
# in the target-aligned coordinate system.
rows = []
for name, model in MODELS.items():
    r, _ = evaluate(
        model, Xs_aligned, y_unsw_train, X_cicids_all, y_cicids,
        "UNSW_to_CICIDS", "CORAL", name
    )
    rows.append(r)

coral_u2c = pd.DataFrame(rows)
display(coral_u2c)


In [ ]:
# ============================================================
# CORAL: CICIDS → UNSW
# ============================================================
# UNSW test FEATURES are used as unlabeled target statistics.
# Their labels remain untouched until evaluation.
X_unsw_align = X_unsw_test

Xs_aligned_rev, coral_stats_c2u = coral_fit_transform(
    X_cicids_align.to_numpy(), X_unsw_align.to_numpy()
)

rows = []
for name, model in MODELS.items():
    r, _ = evaluate(
        model, Xs_aligned_rev, y_cicids_train, X_unsw_test, y_unsw_test,
        "CICIDS_to_UNSW", "CORAL", name
    )
    rows.append(r)

coral_c2u = pd.DataFrame(rows)
display(coral_c2u)


In [ ]:
# ============================================================
# CORAL + class weighting
# ============================================================
rows = []

for name, model in WEIGHTED_MODELS.items():
    r, _ = evaluate(
        model, Xs_aligned, y_unsw_train, X_cicids_all, y_cicids,
        "UNSW_to_CICIDS", "CORAL_class_weighted", name
    )
    rows.append(r)

for name, model in WEIGHTED_MODELS.items():
    r, _ = evaluate(
        model, Xs_aligned_rev, y_cicids_train, X_unsw_test, y_unsw_test,
        "CICIDS_to_UNSW", "CORAL_class_weighted", name
    )
    rows.append(r)

weighted = pd.DataFrame(rows)
display(weighted)


In [ ]:
# ============================================================
# Master comparison + improvement over baseline
# ============================================================
master = pd.concat([baseline, coral_u2c, coral_c2u, weighted], ignore_index=True)

base = baseline[["direction","model","f1","macro_f1","pr_auc","fpr","fnr"]].rename(columns={
    "f1":"baseline_f1","macro_f1":"baseline_macro_f1",
    "pr_auc":"baseline_pr_auc","fpr":"baseline_fpr","fnr":"baseline_fnr"
})

comparison = master.merge(base, on=["direction","model"], how="left")
comparison["delta_f1"] = comparison["f1"] - comparison["baseline_f1"]
comparison["delta_macro_f1"] = comparison["macro_f1"] - comparison["baseline_macro_f1"]
comparison["delta_pr_auc"] = comparison["pr_auc"] - comparison["baseline_pr_auc"]
comparison["delta_fpr"] = comparison["fpr"] - comparison["baseline_fpr"]
comparison["delta_fnr"] = comparison["fnr"] - comparison["baseline_fnr"]

display(comparison.sort_values(["direction","f1"], ascending=[True,False]))

master.to_csv(TABLE_DIR/"01_phase8_master_results.csv", index=False)
comparison.to_csv(TABLE_DIR/"02_phase8_improvement_vs_baseline.csv", index=False)


In [ ]:
# ============================================================
# Covariance mismatch diagnostic
# ============================================================
def cov_dist(X,Y):
    X=np.asarray(X,dtype=np.float64); Y=np.asarray(Y,dtype=np.float64)
    return float(np.linalg.norm(np.cov(X,rowvar=False)-np.cov(Y,rowvar=False), ord="fro"))

diag = pd.DataFrame([
    {
        "direction":"UNSW_to_CICIDS",
        "covariance_distance_before":cov_dist(X_unsw_train,X_cicids_align),
        "covariance_distance_after":cov_dist(Xs_aligned,X_cicids_align)
    },
    {
        "direction":"CICIDS_to_UNSW",
        "covariance_distance_before":cov_dist(X_cicids_align,X_unsw_align),
        "covariance_distance_after":cov_dist(Xs_aligned_rev,X_unsw_align)
    }
])
diag["change"] = diag["covariance_distance_after"]-diag["covariance_distance_before"]
display(diag)
diag.to_csv(TABLE_DIR/"03_coral_covariance_diagnostic.csv",index=False)


In [ ]:
# ============================================================
# Best result per transfer direction
# ============================================================
best = (
    master.sort_values("f1", ascending=False)
    .groupby("direction", as_index=False)
    .first()
)
display(best[[
    "direction","method","model","f1","macro_f1",
    "precision","recall","pr_auc","fpr","fnr"
]])
best.to_csv(TABLE_DIR/"04_best_configuration_by_direction.csv",index=False)

manifest = {
    "phase":"08_domain_adaptation_coral",
    "goal":"Improve cross-dataset NIDS transfer through unsupervised distribution alignment.",
    "random_state":RANDOM_STATE,
    "common_features":COMMON_FEATURES,
    "directions":["UNSW_to_CICIDS","CICIDS_to_UNSW"],
    "methods":["common_baseline","CORAL","CORAL_class_weighted"],
    "models":["Logistic Regression","Random Forest"],
    "cicids_alignment_sample":int(len(cicids_alignment_idx)),
    "target_labels_used_for_alignment":False,
    "target_labels_used_for_model_selection":False
}
with open(OUTPUT_DIR/"phase8_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)

print("Phase 8 complete. Results saved to:", TABLE_DIR)
